In [76]:
import json, math, statistics
from pathlib import Path
import plotly.io as pio
import os
pio.renderers.default = "browser"

# --------- 1) Robust JSON -> list[dict] normalizer (edit here if needed) ---------
def _is_record_list(x):
    return isinstance(x, list) and (len(x) == 0 or isinstance(x[0], dict))

def extract_records(obj):
    """Return a list of dict records from a variety of common result JSON schemas.
    
    Supported patterns (common in benchmark dumps):
      - list[dict]
      - {'results': list[dict]} or {'episodes': list[dict]} or {'data': list[dict]}
      - {'runs': {'methodA': list[dict], ...}}  -> flattened with method key
      - dict-of-dicts keyed by episode_id -> converted to records
    """
    # Case 1: already a list of records
    if _is_record_list(obj):
        return obj

    # Case 2: wrapper keys
    if isinstance(obj, dict):
        for k in ("results", "episodes", "data", "records"):
            if k in obj and _is_record_list(obj[k]):
                return obj[k]

        # Case 3: dict-of-dicts keyed by episode_id
        # e.g., {"ep_0001": {..}, "ep_0002": {..}}
        if all(isinstance(v, dict) for v in obj.values()):
            recs = []
            for key, val in obj.items():
                r = dict(val)
                # keep the key as an identifier if not present
                r.setdefault("episode_id", key)
                recs.append(r)
            return recs

        # Case 4: nested 'runs' or 'methods'
        for k in ("runs", "methods", "agents"):
            if k in obj and isinstance(obj[k], dict):
                recs = []
                for method_name, payload in obj[k].items():
                    subrecs = extract_records(payload)
                    for r in subrecs:
                        rr = dict(r)
                        rr.setdefault("method", method_name)
                        recs.append(rr)
                if recs:
                    return recs

    raise ValueError("Unrecognized JSON schema. Please edit extract_records() to match your file format.")


def load_records(path: str, method_name: str):
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    recs = extract_records(obj)
    # attach method name if missing
    out = []
    for r in recs:
        rr = dict(r)
        rr.setdefault("method", method_name)
        out.append(rr)
    return out


def try_float(x):
    """Convert x to float if possible, else return None."""
    if x is None:
        return None
    if isinstance(x, (int, float)):
        if isinstance(x, bool):
            return None
        return float(x)
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return None
        try:
            return float(s)
        except ValueError:
            return None
    return None


def numeric_keys(records):
    """Infer candidate numeric metric keys from records."""
    keys = set()
    for r in records:
        for k, v in r.items():
            if k in ("method",):
                continue
            fv = try_float(v)
            if fv is not None and not math.isnan(fv) and not math.isinf(fv):
                keys.add(k)
    return sorted(keys)


def summarize_by_method(records, metric_keys):
    """Compute mean, std, count for each metric per method."""
    by_m = {}
    for r in records:
        m = r.get("method", "unknown")
        by_m.setdefault(m, []).append(r)

    summary = {}  # method -> metric -> stats
    for m, rs in by_m.items():
        summary[m] = {}
        for k in metric_keys:
            vals = []
            for r in rs:
                fv = try_float(r.get(k))
                if fv is None or math.isnan(fv) or math.isinf(fv):
                    continue
                vals.append(fv)
            if len(vals) == 0:
                continue
            mu = statistics.fmean(vals)
            sd = statistics.pstdev(vals) if len(vals) > 1 else 0.0
            summary[m][k] = {"mean": mu, "std": sd, "n": len(vals)}
    return summary


In [78]:
# --------- 2) Load & merge all results in ONE block ---------
base = os.path.expanduser("~/lighthouse/home/junzhe/Projects/SG-VLN")
poliformer_path = os.path.join(base, "dump2/benchmark_poliformer_jan28/result.json")
uninavid_path   = os.path.join(base, "dump2/benchmark_uninavid_jan28/result.json")
modular_agent_path   = os.path.join(base, "dump2/benchmark_agent_jan29/result.json")

records = []
records += load_records(poliformer_path, method_name="poliformer")
records += load_records(uninavid_path, method_name="uninavid")
records += load_records(modular_agent_path,   method_name="modular_agent")
print("Loaded records:", len(records))

# Infer numeric metric keys (you can override manually if you want)
metric_keys = numeric_keys(records)
summary = summarize_by_method(records, metric_keys)

# Pretty-print a small summary preview
for m, mets in summary.items():
    print("\n==", m, "==")
    for k in sorted(mets.keys())[:10]:
        s = mets[k]
        print(f"  {k}: mean={s['mean']:.4g}, std={s['std']:.4g}, n={s['n']}")


Loaded records: 982

== poliformer ==
  closest_goal: mean=0.1433, std=0.5031, n=307
  distance_to_goal: mean=17.53, std=17.11, n=307
  episode_duration: mean=623.3, std=883, n=307
  episode_id: mean=9.378, std=5.863, n=307
  oracle_navigation_error: mean=14.04, std=14.5, n=307
  oracle_success: mean=0.07166, std=0.2579, n=307
  path_length: mean=57.38, std=53.94, n=307
  sim_duration: mean=147, std=125.7, n=307
  spl: mean=0, std=0, n=307
  step: mean=165.6, std=138.9, n=307

== uninavid ==
  closest_goal: mean=0.1523, std=0.5305, n=302
  distance_to_goal: mean=17.21, std=19.7, n=302
  episode_duration: mean=631.2, std=759.7, n=302
  episode_id: mean=9.394, std=5.816, n=302
  oracle_navigation_error: mean=12.94, std=14.97, n=302
  oracle_success: mean=0.1126, std=0.3161, n=302
  path_length: mean=40.88, std=47.18, n=302
  sim_duration: mean=116.9, std=122.2, n=302
  spl: mean=0, std=0, n=302
  step: mean=71.09, std=67.21, n=302

== modular_agent ==
  closest_goal: mean=0.134, std=0.49

In [47]:
# Calculate the success rate (final distance to goal smaller than threshold)
# 1) pick which field is "distance to goal"
DIST_CANDIDATES = ["distance_to_goal"]

def get_distance_value(rec):
    # direct keys
    for k in DIST_CANDIDATES:
        if k in rec and rec[k] is not None:
            try:
                return float(rec[k])
            except (TypeError, ValueError):
                pass
    # nested dicts (common patterns)
    for parent in ("metrics", "metric", "stats", "result", "eval", "episode"):
        obj = rec.get(parent)
        if isinstance(obj, dict):
            for k in DIST_CANDIDATES:
                if k in obj and obj[k] is not None:
                    try:
                        return float(obj[k])
                    except (TypeError, ValueError):
                        pass
    return None

# 2) compute new success metric
THRESH = 1.6
missing_dist = 0
for r in records:
    d = get_distance_value(r)
    if d is None:
        missing_dist += 1
        r["success_1p5"] = None   # keep as None so it won't affect mean/std
        r["dist_used_for_success"] = None
    else:
        r["success_1p5"] = 1.0 if d < THRESH else 0.0
        r["dist_used_for_success"] = d

print(f"Computed success_1p5 with threshold {THRESH}. Missing distance in {missing_dist}/{len(records)} records.")

# 3) summarize ONLY this metric (simpler & faster)
metric_keys = ["success_1p5"]
summary = summarize_by_method(records, metric_keys)

print("Methods found:", list(summary.keys()))
for m, mets in summary.items():
    s = mets["success_1p5"]
    print(f"\n== {m} ==")
    print(f"  success_1p5 (dist< {THRESH}): mean={s['mean']:.4g}, std={s['std']:.4g}, n={s['n']}")


Computed success_1p5 with threshold 1.6. Missing distance in 0/885 records.
Methods found: ['poliformer', 'uninavid', 'modular_agent']

== poliformer ==
  success_1p5 (dist< 1.6): mean=0.04027, std=0.1966, n=298

== uninavid ==
  success_1p5 (dist< 1.6): mean=0.09934, std=0.2991, n=302

== modular_agent ==
  success_1p5 (dist< 1.6): mean=0.05263, std=0.2233, n=285


In [4]:
# --------- 0) Build scenes list (episode json filenames) ----------
import os, json
import numpy as np

episode_folder = "robot_env/episodes/"

scenes = []
for fn in os.listdir(episode_folder):
    # keep only json episode files, skip helper files like checked_gr.txt, checked_vc.txt, test*.json
    if fn.endswith(".json") and ("test" not in fn.lower()):
        scenes.append(fn)
scenes = sorted(scenes)

print("Found scene files:", len(scenes))
print("Example:", scenes[:5])


Found scene files: 38
Example: ['grCommercial_scene1.json', 'grCommercial_scene11.json', 'grCommercial_scene13.json', 'grCommercial_scene2.json', 'grCommercial_scene26.json']


In [18]:
# Get the episode shortest goal distance
import os, json
import numpy as np
import plotly.express as px

episode_folder = "robot_env/episodes/"
episode_length = {}

for scene in scenes:
    data = json.load(open(os.path.join(episode_folder, scene)))
    for episode in data:
        scene_id = episode["scene_id"]
        episode_label = f"{scene_id}_{episode['episode_id']}"
        closest_goal_idx = episode["closest_goal_idx"]
        closest_goal = episode["goals"][closest_goal_idx]
        path_length = closest_goal["path_length"]
        episode_length[episode_label] = path_length

In [48]:
# Quantile-binned grouped bar + rolling trend line (no pandas)
import numpy as np
import plotly.graph_objects as go

THRESH = 1.6
NBINS = 20

def get_method(r):
    return r.get("method") or r.get("method_name") or r.get("model") or "unknown"

# --- 1) collect per-method (shortest_goal_len, success) ---
data = {}
missing = {"no_ep_label": 0, "no_shortest": 0, "no_dist": 0, "bad_cast": 0}

for r in records:
    ep = r.get("episode_label")
    if ep is None:
        missing["no_ep_label"] += 1
        continue

    shortest = episode_length.get(ep)      # <-- shortest goal path length from episodes json
    d = r.get("distance_to_goal", None)    # <-- final distance

    if shortest is None:
        missing["no_shortest"] += 1
        continue
    if d is None:
        missing["no_dist"] += 1
        continue

    try:
        shortest = float(shortest)
        d = float(d)
    except Exception:
        missing["bad_cast"] += 1
        continue

    m = get_method(r)
    data.setdefault(m, {"x": [], "succ": []})
    data[m]["x"].append(shortest)
    data[m]["succ"].append(1.0 if d < THRESH else 0.0)

methods = sorted(data.keys())
print("Methods:", methods)
print("Missing:", missing)
for m in methods:
    x = np.array(data[m]["x"], dtype=float)
    s = np.array(data[m]["succ"], dtype=float)
    print(f"{m}: points={len(x)}, overall_success={float(np.mean(s)) if len(s) else 'N/A'}")

# --- 2) build global quantile bin edges (equal-count bins across all methods) ---
all_x = np.array([v for m in methods for v in data[m]["x"]], dtype=float)
all_x = all_x[np.isfinite(all_x)]
if len(all_x) == 0:
    raise RuntimeError("No valid shortest lengths found to plot.")

# quantile edges, then unique-sort (handles ties)
qs = np.linspace(0, 1, NBINS + 1)
edges = np.quantile(all_x, qs)
edges = np.unique(edges)

# if too many ties -> fewer effective bins
if len(edges) < 3:
    raise RuntimeError("Quantile edges collapsed (too many identical lengths). Try fewer NBINS or inspect data.")

# helper: assign bin id by edges
def bin_index(values, edges):
    # edges length K => bins = K-1
    idx = np.digitize(values, edges, right=False) - 1
    return np.clip(idx, 0, len(edges) - 2)

bin_centers = 0.5 * (edges[:-1] + edges[1:])
bin_left = edges[:-1]
bin_right = edges[1:]
nb_eff = len(bin_centers)

# --- 3) grouped bar: success rate per quantile-bin, per method ---
fig = go.Figure()

for m in methods:
    x = np.array(data[m]["x"], dtype=float)
    s = np.array(data[m]["succ"], dtype=float)
    idx = bin_index(x, edges)

    rates = np.full(nb_eff, np.nan, dtype=float)
    counts = np.zeros(nb_eff, dtype=int)

    for i in range(nb_eff):
        mask = (idx == i)
        c = int(mask.sum())
        counts[i] = c
        if c > 0:
            rates[i] = float(np.mean(s[mask]))

    valid = np.isfinite(rates)

    fig.add_trace(go.Bar(
        name=f"{m} (binned)",
        x=bin_centers[valid],
        y=rates[valid],
        customdata=np.stack([counts[valid], bin_left[valid], bin_right[valid]], axis=1),
        hovertemplate=(
            "bin=[%{customdata[1]:.2f}, %{customdata[2]:.2f}]<br>"
            "center=%{x:.2f}<br>"
            "success_rate=%{y:.3f}<br>"
            "count=%{customdata[0]:.0f}<extra></extra>"
        ),
        opacity=0.75,
    ))

# --- 4) rolling trend line per method (sorted by shortest length) ---
def rolling_mean(y, w):
    if w <= 1:
        return y.copy()
    # centered moving average via convolution (pad edges with reflection)
    w = int(w)
    pad = w // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    kernel = np.ones(w, dtype=float) / w
    return np.convolve(ypad, kernel, mode="valid")

for m in methods:
    x = np.array(data[m]["x"], dtype=float)
    s = np.array(data[m]["succ"], dtype=float)

    order = np.argsort(x)
    x_sorted = x[order]
    s_sorted = s[order]

    # auto window ~ 1/20 of data, min 30, max 400 (tweak if you want)
    ROLL_W = int(max(30, min(400, len(s_sorted) // 20)))
    y_roll = rolling_mean(s_sorted, ROLL_W)

    fig.add_trace(go.Scatter(
        name=f"{m} (rolling w={ROLL_W})",
        x=x_sorted,
        y=y_roll,
        mode="lines",
        line=dict(width=3),
        hovertemplate="shortest=%{x:.2f}<br>rolling_success=%{y:.3f}<extra></extra>",
    ))

fig.update_layout(
    # barmode="group",
    title=f"Success vs shortest goal path length (quantile bins={NBINS}) + rolling trend   (success: distance_to_goal < {THRESH})",
    xaxis_title="shortest goal path length (episode_length[episode_label])",
    yaxis_title="success rate",
    yaxis=dict(range=[0, 0.8]),
    xaxis=dict(range=[0, 100]),
)

fig  # show in ipynb


Methods: ['modular_agent', 'poliformer', 'uninavid']
Missing: {'no_ep_label': 0, 'no_shortest': 0, 'no_dist': 0, 'bad_cast': 0}
modular_agent: points=285, overall_success=0.05263157894736842
poliformer: points=298, overall_success=0.040268456375838924
uninavid: points=302, overall_success=0.09933774834437085


In [36]:
# --- Two separate figures: PoliFormer vs UniNavID (success/failure counts in 10 bins) ---
import numpy as np
import plotly.graph_objects as go

pio.renderers.default = "notebook_connected"

THRESH = 1.6
NBINS = 10

def get_method(r):
    return r.get("method") or r.get("method_name") or r.get("model") or "unknown"

# 1) collect per method: (shortest_len, is_success)
data = {}
for r in records:
    ep = r.get("episode_label")
    if ep is None:
        continue

    shortest = episode_length.get(ep)      # shortest goal length
    d = r.get("distance_to_goal", None)    # final dist

    if shortest is None or d is None:
        continue

    try:
        shortest = float(shortest)
        d = float(d)
    except Exception:
        continue

    m = get_method(r)
    data.setdefault(m, {"x": [], "succ": []})
    data[m]["x"].append(shortest)
    data[m]["succ"].append(1 if d < THRESH else 0)

methods = sorted(data.keys())
print("Methods found:", methods)

# 2) helper to make one figure for one method
def make_fig_for_method(method_name, x_list, succ_list):
    x = np.array(x_list, dtype=float)
    s = np.array(succ_list, dtype=int)

    if len(x) == 0:
        print(f"[{method_name}] no points to plot.")
        return None

    xmin, xmax = float(np.min(x)), float(np.max(x))
    edges = np.linspace(xmin, xmax, NBINS + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    left = edges[:-1]
    right = edges[1:]

    idx = np.clip(np.digitize(x, edges) - 1, 0, NBINS - 1)

    succ_counts = np.zeros(NBINS, dtype=int)
    fail_counts = np.zeros(NBINS, dtype=int)

    for i in range(NBINS):
        mask = (idx == i)
        if mask.any():
            succ_counts[i] = int(np.sum(s[mask] == 1))
            fail_counts[i] = int(np.sum(s[mask] == 0))

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="success",
        x=centers,
        y=succ_counts,
        customdata=np.stack([left, right, succ_counts, fail_counts], axis=1),
        hovertemplate=(
            "bin=[%{customdata[0]:.2f}, %{customdata[1]:.2f}]<br>"
            "success=%{customdata[2]:.0f}<br>"
            "failure=%{customdata[3]:.0f}<extra></extra>"
        ),
        opacity=0.85,
    ))
    fig.add_trace(go.Bar(
        name="failure",
        x=centers,
        y=fail_counts,
        customdata=np.stack([left, right, succ_counts, fail_counts], axis=1),
        hovertemplate=(
            "bin=[%{customdata[0]:.2f}, %{customdata[1]:.2f}]<br>"
            "success=%{customdata[2]:.0f}<br>"
            "failure=%{customdata[3]:.0f}<extra></extra>"
        ),
        opacity=0.55,
    ))

    fig.update_layout(
        barmode="group",
        title=f"{method_name}: Success/Failure counts vs shortest goal path length (NBINS={NBINS}, success: d<{THRESH})",
        xaxis_title="shortest goal path length (episode_length[episode_label])",
        yaxis_title="count",
    )
    return fig

# 3) make and display one fig per method (two plots)
# If your method names are exactly "poliformer" and "uninavid", this will show two figs.
for m in methods:
    fig = make_fig_for_method(m, data[m]["x"], data[m]["succ"])
    if fig is not None:
        fig.show()  # display in ipynb


Methods found: ['modular_agent', 'poliformer', 'uninavid']


In [49]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

THRESH = 1.6
NBINS = 10

pio.renderers.default = "notebook_connected"
def make_fig_for_method(method_name, x_list, succ_list):
    x = np.array(x_list, dtype=float)
    s = np.array(succ_list, dtype=int)

    if len(x) == 0:
        print(f"[{method_name}] no points to plot.")
        return None

    xmin, xmax = float(np.min(x)), float(np.max(x))
    edges = np.linspace(xmin, xmax, NBINS + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    left = edges[:-1]
    right = edges[1:]

    idx = np.clip(np.digitize(x, edges) - 1, 0, NBINS - 1)

    succ_counts = np.zeros(NBINS, dtype=int)
    fail_counts = np.zeros(NBINS, dtype=int)
    rates = np.full(NBINS, np.nan, dtype=float)
    totals = np.zeros(NBINS, dtype=int)

    for i in range(NBINS):
        mask = (idx == i)
        c = int(mask.sum())
        totals[i] = c
        if c > 0:
            sc = int(np.sum(s[mask] == 1))
            fc = int(np.sum(s[mask] == 0))
            succ_counts[i] = sc
            fail_counts[i] = fc
            rates[i] = sc / (sc + fc) if (sc + fc) > 0 else np.nan

    # optional: hide empty bins so the curve doesn't look weird
    valid = (totals > 0)
    xc = centers[valid]
    scv = succ_counts[valid]
    fcv = fail_counts[valid]
    rv = rates[valid]
    lv = left[valid]
    rv_edge = right[valid]
    tv = totals[valid]

    # two y-axes: left=rate, right=counts
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # bars on RIGHT axis
    fig.add_trace(go.Bar(
        name="success (count)",
        x=xc,
        y=scv,
        customdata=np.stack([lv, rv_edge, scv, fcv, tv, rv], axis=1),
        hovertemplate=(
            "bin=[%{customdata[0]:.2f}, %{customdata[1]:.2f}]<br>"
            "success=%{customdata[2]:.0f}<br>"
            "failure=%{customdata[3]:.0f}<br>"
            "total=%{customdata[4]:.0f}<br>"
            "success_rate=%{customdata[5]:.3f}<extra></extra>"
        ),
        opacity=0.85,
    ), secondary_y=True)

    fig.add_trace(go.Bar(
        name="failure (count)",
        x=xc,
        y=fcv,
        customdata=np.stack([lv, rv_edge, scv, fcv, tv, rv], axis=1),
        hovertemplate=(
            "bin=[%{customdata[0]:.2f}, %{customdata[1]:.2f}]<br>"
            "success=%{customdata[2]:.0f}<br>"
            "failure=%{customdata[3]:.0f}<br>"
            "total=%{customdata[4]:.0f}<br>"
            "success_rate=%{customdata[5]:.3f}<extra></extra>"
        ),
        opacity=0.55,
    ), secondary_y=True)

    # rate line on LEFT axis
    fig.add_trace(go.Scatter(
        name="success_rate",
        x=xc,
        y=rv,
        mode="lines+markers",
        hovertemplate="bin_center=%{x:.2f}<br>success_rate=%{y:.3f}<extra></extra>",
    ), secondary_y=False)

    fig.update_layout(
        barmode="group",
        title=f"{method_name}: Success rate (left) + success/failure counts (right) vs shortest goal length (NBINS={NBINS}, success: d<{THRESH})",
        xaxis_title="shortest goal path length (episode_length[episode_label])",
        legend_title="",
    )

    fig.update_yaxes(title_text="success_rate", range=[0, 0.6], secondary_y=False)
    fig.update_yaxes(title_text="count", secondary_y=True)

    return fig

# show two figures (one per method)
for m in methods:
    fig = make_fig_for_method(m, data[m]["x"], data[m]["succ"])
    if fig is not None:
        fig.show()


In [ ]:
import numpy as np

THRESH = 1.6  # success = distance_to_goal < THRESH

def get_split_from_episode_label(ep_label: str):
    """Map episode_label prefix to split name."""
    if not isinstance(ep_label, str):
        return None
    s = ep_label.strip().lower()
    if s.startswith("gr"):
        return "indoor"
    if s.startswith("vc"):
        return "outdoor"
    if s.startswith("innout"):
        return "innout"
    return "other"   # 你也可以改成 None 直接丢弃

def safe_float(v):
    try:
        if v is None: 
            return None
        if isinstance(v, bool):
            return None
        return float(v)
    except Exception:
        return None

# --- group by (method, split) ---
stats = {}  # (method, split) -> {"succ": [], "spl": []}
missing = {"no_episode_label": 0, "no_dist": 0, "no_spl": 0}

for r in records:
    m = r.get("method", "unknown")

    ep = r.get("episode_label", None)
    if ep is None:
        missing["no_episode_label"] += 1
        continue

    split = get_split_from_episode_label(ep)
    # 如果你不想要 other：把下面两行打开
    # if split == "other":
    #     continue

    d = safe_float(r.get("distance_to_goal"))
    if d is None:
        missing["no_dist"] += 1
        continue

    spl = safe_float(r.get("spl"))
    if spl is None:
        missing["no_spl"] += 1
        continue

    key = (m, split)
    stats.setdefault(key, {"succ": [], "spl": []})

    succ = 1.0 if d < THRESH else 0.0
    stats[key]["succ"].append(succ)
    stats[key]["spl"].append(spl)

print("Missing counts:", missing)

# --- print results in a readable way ---
methods = sorted(set(k[0] for k in stats.keys()))
splits = ["indoor", "outdoor", "innout", "other"]

for m in methods:
    print("\n==============================")
    print("METHOD:", m)
    for sp in splits:
        key = (m, sp)
        if key not in stats:
            continue
        succ_arr = np.array(stats[key]["succ"], dtype=float)
        spl_arr  = np.array(stats[key]["spl"], dtype=float)

        succ_rate = float(np.mean(succ_arr)) if len(succ_arr) else float("nan")
        spl_mean  = float(np.mean(spl_arr))  if len(spl_arr)  else float("nan")

        print(f"  [{sp:7s}]  n={len(succ_arr):4d}   success_rate={succ_rate:.4f}   mean_spl={spl_mean:.4f}")

Missing counts: {'no_episode_label': 0, 'no_dist': 0, 'no_spl': 0}

METHOD: modular_agent
  [indoor ]  n= 123   success_rate=0.0650   mean_spl=0.0000
  [outdoor]  n= 147   success_rate=0.0476   mean_spl=0.0000
  [innout ]  n=   3   success_rate=0.0000   mean_spl=0.0000

METHOD: poliformer
  [indoor ]  n= 148   success_rate=0.0811   mean_spl=0.0000
  [outdoor]  n= 150   success_rate=0.0000   mean_spl=0.0000

METHOD: uninavid
  [indoor ]  n= 150   success_rate=0.1133   mean_spl=0.0000
  [outdoor]  n= 152   success_rate=0.0855   mean_spl=0.0000


In [73]:
# table 2
import numpy as np

THRESH = 1.6  # success = distance_to_goal < THRESH

def get_method(r):
    return r.get("method") or r.get("method_name") or r.get("model") or "unknown"

def get_split_from_episode_label(ep_label: str):
    """Map episode_label prefix to split name."""
    if not isinstance(ep_label, str):
        return None
    s = ep_label.strip().lower()
    if s.startswith("gr"):
        return "indoor"
    if s.startswith("vc"):
        return "outdoor"
    if s.startswith("innout"):
        return "innout"
    return "other"   # or return None to drop

def get_nav_type_from_episode_label(ep_label: str):
    """
    object navigation if contains '_store' (e.g., vc_barcelona_store_15),
    else place navigation.
    """
    if not isinstance(ep_label, str):
        return None
    s = ep_label.strip().lower()
    if "_store" in s:
        return "place"
    return "object"

INCLUDE_OTHER_IN_ALL = False

# --- group by (method, split, nav_type) ---
# stats[(m, split, nav_type)] = {"succ": [...], "spl": [...]}
stats = {}
missing = {
    "no_episode_label": 0,
    "no_dist": 0,
    "no_shortest": 0,
    "bad_cast": 0,
}

for r in records:
    m = get_method(r)

    ep = r.get("episode_label", None)
    if ep is None:
        missing["no_episode_label"] += 1
        continue

    split = get_split_from_episode_label(ep)
    nav_type = get_nav_type_from_episode_label(ep)

    # final distance to goal (same helper you already have)
    d = get_distance_value(r)
    if d is None:
        missing["no_dist"] += 1
        continue

    # L* from episodes json; L from result.json
    L_star   = episode_length.get(ep)        # shortest path length (L*)
    L_actual = r.get("path_length", None)    # actual traveled length (L)

    if L_star is None or L_actual is None:
        missing["no_shortest"] += 1
        continue

    try:
        d = float(d)
        L_star = float(L_star)
        L_actual = float(L_actual)
    except Exception:
        missing["bad_cast"] += 1
        continue

    succ = 1.0 if d < THRESH else 0.0
    spl_value = (L_star / max(L_star, L_actual)) if succ > 0 else 0.0

    key = (m, split, nav_type)
    stats.setdefault(key, {"succ": [], "spl": []})
    stats[key]["succ"].append(succ)
    stats[key]["spl"].append(spl_value)

print("Missing counts:", missing)

# --- helpers ---
def desc(arr):
    a = np.array(arr, dtype=float)
    if a.size == 0:
        return (0, float("nan"), float("nan"), float("nan"), float("nan"))
    return (int(a.size), float(np.mean(a)), float(np.std(a)), float(np.min(a)), float(np.max(a)))

# --- printing ---
methods = sorted(set(k[0] for k in stats.keys()))
splits = ["indoor", "outdoor", "innout"]      # main splits
nav_types = ["place", "object"]              # two tasks

def collect_all(m, nt):
    """Merge indoor/outdoor/innout (and optionally other) for (method, nav_type)."""
    succ_all = []
    spl_all = []
    for sp in (splits + (["other"] if INCLUDE_OTHER_IN_ALL else [])):
        key = (m, sp, nt)
        if key in stats:
            succ_all.extend(stats[key]["succ"])
            spl_all.extend(stats[key]["spl"])
    return succ_all, spl_all

for m in methods:
    print("\n==============================")
    print("METHOD:", m)

    # 1) per split
    for sp in splits:
        for nt in nav_types:
            key = (m, sp, nt)
            if key not in stats:
                continue

            succ_arr = np.array(stats[key]["succ"], dtype=float)
            succ_rate = float(np.mean(succ_arr)) if succ_arr.size else float("nan")
            n1, mu1, sd1, mn1, mx1 = desc(stats[key]["spl"])

            print(f"  [{sp:7s} | {nt:6s}]  n={succ_arr.size:4d}   success_rate={succ_rate:.4f}  (d<{THRESH})")
            print(f"                  spl      mean      = {mu1:.6f} ")
            # print(f"                  spl      mean/std/min/max      = {mu1:.6f} / {sd1:.6f} / {mn1:.6f} / {mx1:.6f}")

    # 2) ALL SCENES (merged) for each nav_type
    print("  -------- ALL SCENES (merged) --------")
    for nt in nav_types:
        succ_all, spl_all = collect_all(m, nt)
        succ_all = np.array(succ_all, dtype=float)
        succ_rate_all = float(np.mean(succ_all)) if succ_all.size else float("nan")
        n1, mu1, sd1, mn1, mx1 = desc(spl_all)

        print(f"  [all     | {nt:6s}]  n={succ_all.size:4d}   success_rate={succ_rate_all:.4f}  (d<{THRESH})")
        print(f"                  spl      mean      = {mu1:.6f} ")
        # print(f"                  spl      mean/std/min/max      = {mu1:.6f} / {sd1:.6f} / {mn1:.6f} / {mx1:.6f}")

Missing counts: {'no_episode_label': 0, 'no_dist': 0, 'no_shortest': 0, 'bad_cast': 0}

METHOD: modular_agent
  [indoor  | object]  n= 124   success_rate=0.0645  (d<1.6)
                  spl      mean      = 0.020054 
  [outdoor | place ]  n=  68   success_rate=0.1029  (d<1.6)
                  spl      mean      = 0.021556 
  [outdoor | object]  n=  82   success_rate=0.0000  (d<1.6)
                  spl      mean      = 0.000000 
  [innout  | object]  n=  11   success_rate=0.0000  (d<1.6)
                  spl      mean      = 0.000000 
  -------- ALL SCENES (merged) --------
  [all     | place ]  n=  68   success_rate=0.1029  (d<1.6)
                  spl      mean      = 0.021556 
  [all     | object]  n= 217   success_rate=0.0369  (d<1.6)
                  spl      mean      = 0.011460 

METHOD: poliformer
  [indoor  | object]  n= 148   success_rate=0.0811  (d<1.6)
                  spl      mean      = 0.029231 
  [outdoor | place ]  n=  68   success_rate=0.0000  (d<1.6)
       

In [71]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# ----------------------------
# Rendering
# ----------------------------
pio.renderers.default = "notebook_connected"

# ----------------------------
# Config
# ----------------------------
THRESH = 1.6
NBINS = 10
X_MIN, X_MAX = 0.0, 50.0  # 你要求 episode length 显示 0~50（并固定 bins）

# ----------------------------
# Helpers: method / split / nav type
# ----------------------------
def get_method(r):
    return r.get("method") or r.get("method_name") or r.get("model") or "unknown"

def get_split_from_episode_label(ep_label: str):
    if not isinstance(ep_label, str):
        return None
    s = ep_label.strip().lower()
    if s.startswith("gr"):
        return "indoor"
    if s.startswith("vc"):
        return "outdoor"
    if s.startswith("innout"):
        return "innout"
    return "other"

def get_nav_type_from_episode_label(ep_label: str):
    if not isinstance(ep_label, str):
        return None
    s = ep_label.strip().lower()
    # your rule: contains "_store" -> place nav
    if "_store" in s:
        return "place"
    return "object"

# ----------------------------
# Collect per (method, split, nav_type)
# Required existing:
# - records: list[dict]
# - episode_length: dict[str -> L*]
# - get_distance_value(r)
# ----------------------------
data = {}  # key=(method, split, nav_type) -> dict with lists
missing = {"no_ep": 0, "no_d": 0, "no_Lstar": 0, "no_Lactual": 0, "bad_cast": 0, "x_outside_0_50": 0}

for r in records:
    ep = r.get("episode_label")
    if ep is None:
        missing["no_ep"] += 1
        continue

    m = get_method(r)
    sp = get_split_from_episode_label(ep)
    nt = get_nav_type_from_episode_label(ep)

    d = get_distance_value(r)
    if d is None:
        missing["no_d"] += 1
        continue

    L_star = episode_length.get(ep)
    if L_star is None:
        missing["no_Lstar"] += 1
        continue

    L_actual = r.get("path_length", None)
    if L_actual is None:
        missing["no_Lactual"] += 1
        continue

    try:
        d = float(d)
        L_star = float(L_star)
        L_actual = float(L_actual)
    except Exception:
        missing["bad_cast"] += 1
        continue

    # 只统计 0~50 的 episode_length（否则你的 x 轴固定 0~50 会把它们挤到边缘/看不到）
    if not (X_MIN <= L_star <= X_MAX):
        missing["x_outside_0_50"] += 1
        continue

    succ = 1 if d < THRESH else 0
    spl = (L_star / max(L_star, L_actual)) if succ else 0.0

    key = (m, sp, nt)
    data.setdefault(key, {"x": [], "succ": [], "spl": [], "L_actual": []})
    data[key]["x"].append(L_star)
    data[key]["succ"].append(succ)
    data[key]["spl"].append(spl)
    data[key]["L_actual"].append(L_actual)

print("Missing:", missing)

methods = sorted(set(k[0] for k in data.keys()))
if len(methods) == 0:
    raise RuntimeError("No methods found in data. Check records/method field parsing.")

# ----------------------------
# Color map: one color per method (尽量差异大)
# ----------------------------
PALETTE = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown
    "#e377c2",  # pink
    "#7f7f7f",  # gray
    "#bcbd22",  # olive
    "#17becf",  # cyan
]
method_color = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(methods)}

# ----------------------------
# Merge all splits for (method, nav_type)
# ----------------------------
def merged_all_for_method_nav(method_name, nav_type):
    x, succ, spl, L_actual = [], [], [], []
    for (m, sp, nt), v in data.items():
        if m != method_name or nt != nav_type:
            continue
        x.extend(v["x"])
        succ.extend(v["succ"])
        spl.extend(v["spl"])
        L_actual.extend(v["L_actual"])
    return {"x": x, "succ": succ, "spl": spl, "L_actual": L_actual}

# ----------------------------
# Bin stats with FIXED edges [0,50]
# ----------------------------
FIXED_EDGES = np.linspace(X_MIN, X_MAX, NBINS + 1)

def bin_stats_fixed_edges(x, succ, val=None, edges=FIXED_EDGES):
    x = np.array(x, dtype=float)
    s = np.array(succ, dtype=int)
    if val is not None:
        v = np.array(val, dtype=float)

    left = edges[:-1]
    right = edges[1:]
    centers = 0.5 * (left + right)
    nbins = len(left)

    # assign to bins
    idx = np.digitize(x, edges) - 1
    idx = np.clip(idx, 0, nbins - 1)

    succ_counts = np.zeros(nbins, dtype=int)
    fail_counts = np.zeros(nbins, dtype=int)
    totals = np.zeros(nbins, dtype=int)
    rates = np.full(nbins, np.nan, dtype=float)
    means = np.full(nbins, np.nan, dtype=float) if val is not None else None

    for i in range(nbins):
        mask = (idx == i)
        c = int(mask.sum())
        totals[i] = c
        if c > 0:
            sc = int(np.sum(s[mask] == 1))
            fc = int(np.sum(s[mask] == 0))
            succ_counts[i] = sc
            fail_counts[i] = fc
            rates[i] = sc / (sc + fc) if (sc + fc) > 0 else np.nan
            if val is not None:
                means[i] = float(np.mean(v[mask]))

    valid = totals > 0
    out = {
        "centers": centers[valid],
        "left": left[valid],
        "right": right[valid],
        "succ_counts": succ_counts[valid],
        "fail_counts": fail_counts[valid],
        "totals": totals[valid],
        "rates": rates[valid],
        "bin_width": float(np.median(right - left)),  # fixed -> constant
    }
    if val is not None:
        out["means"] = means[valid]
    return out

# ----------------------------
# Situations (5 figs)
# ----------------------------
SITUATIONS = [
    ("indoor", "object"),
    ("indoor", "place"),
    ("outdoor", "object"),
    ("outdoor", "place"),
    ("all", "merged"),
]

def make_combined_fig_for_situation(sp, nt):
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        specs=[[{"secondary_y": True}], [{}], [{}]],
        vertical_spacing=0.08,
        subplot_titles=[
            "Counts (stacked: success+failure on right) + Success Rate (left)",
            "Mean SPL per Bin",
            "Mean Actual Length per Bin",
        ],
    )

    any_data = False

    # 固定 bin width（0~50 / NBINS）
    bin_w = (X_MAX - X_MIN) / NBINS

    # 你说“柱子宽度不错但叠一起了”：解决方案 = offset_step >= BAR_WIDTH + gap
    # BAR_WIDTH 用 bin_w 的比例；offset_step 用 BAR_WIDTH + gap
    BAR_WIDTH = 0.28 * bin_w          # 你现在觉得宽度不错，可以保持 0.25~0.35
    INNER_GAP = 0.10 * bin_w          # 方法之间留白（调大就更分开）
    offset_step = BAR_WIDTH + INNER_GAP

    # method offsets: centered around 0
    M = len(methods)
    offsets = np.array([(i - (M - 1) / 2.0) * offset_step for i in range(M)], dtype=float)

    for mi, m in enumerate(methods):
        color = method_color[m]

        if sp == "all":
            merged_place = merged_all_for_method_nav(m, "place")
            merged_object = merged_all_for_method_nav(m, "object")
            grp = {
                "x": merged_place["x"] + merged_object["x"],
                "succ": merged_place["succ"] + merged_object["succ"],
                "spl": merged_place["spl"] + merged_object["spl"],
                "L_actual": merged_place["L_actual"] + merged_object["L_actual"],
            }
        else:
            key = (m, sp, nt)
            if key not in data:
                continue
            grp = data[key]

        if len(grp["x"]) == 0:
            continue

        any_data = True

        b0 = bin_stats_fixed_edges(grp["x"], grp["succ"])
        b1 = bin_stats_fixed_edges(grp["x"], grp["succ"], grp["spl"])
        b2 = bin_stats_fixed_edges(grp["x"], grp["succ"], grp["L_actual"])

        # 注意：offset 必须是 episode_length 的单位（x 轴单位），不能用 0.12 这种小数
        xc = b0["centers"] + offsets[mi]

        cd0 = np.stack([b0["left"], b0["right"], b0["succ_counts"], b0["fail_counts"], b0["totals"], b0["rates"]], axis=1)

        # Row 1: stacked bars (success + failure) on secondary y
        fig.add_trace(go.Bar(
            name=f"{m} success",
            x=xc, y=b0["succ_counts"],
            width=BAR_WIDTH,
            marker_color=color,
            opacity=0.85,
            customdata=cd0,
            hovertemplate=(
                f"method={m}<br>"
                "bin=[%{customdata[0]:.1f}, %{customdata[1]:.1f}]<br>"
                "success=%{customdata[2]:.0f}<br>"
                "failure=%{customdata[3]:.0f}<br>"
                "total=%{customdata[4]:.0f}<br>"
                "success_rate=%{customdata[5]:.3f}<extra></extra>"
            ),
        ), row=1, col=1, secondary_y=True)

        fig.add_trace(go.Bar(
            name=f"{m} failure",
            x=xc, y=b0["fail_counts"],
            width=BAR_WIDTH,
            marker_color=color,
            opacity=0.35,   # 同色更淡，表示 failure
            customdata=cd0,
            hovertemplate=(
                f"method={m}<br>"
                "bin=[%{customdata[0]:.1f}, %{customdata[1]:.1f}]<br>"
                "success=%{customdata[2]:.0f}<br>"
                "failure=%{customdata[3]:.0f}<br>"
                "total=%{customdata[4]:.0f}<br>"
                "success_rate=%{customdata[5]:.3f}<extra></extra>"
            ),
        ), row=1, col=1, secondary_y=True)

        # Row 1: success rate line on primary y (same color)
        fig.add_trace(go.Scatter(
            name=f"{m} success_rate",
            x=xc, y=b0["rates"],
            mode="lines+markers",
            line=dict(color=color),
            marker=dict(color=color),
            hovertemplate=f"method={m}<br>bin_center=%{{x:.1f}}<br>success_rate=%{{y:.3f}}<extra></extra>",
        ), row=1, col=1, secondary_y=False)

        # Row 2: mean SPL bar (same color)
        cd1 = np.stack([b1["left"], b1["right"], b1["totals"], b1["means"]], axis=1)
        fig.add_trace(go.Bar(
            name=f"{m} mean_spl",
            x=xc, y=b1["means"],
            width=BAR_WIDTH,
            marker_color=color,
            opacity=0.75,
            customdata=cd1,
            hovertemplate=(
                f"method={m}<br>"
                "bin=[%{customdata[0]:.1f}, %{customdata[1]:.1f}]<br>"
                "total=%{customdata[2]:.0f}<br>"
                "mean_spl=%{customdata[3]:.3f}<extra></extra>"
            ),
        ), row=2, col=1)

        # Row 3: mean L_actual bar (same color)
        cd2 = np.stack([b2["left"], b2["right"], b2["totals"], b2["means"]], axis=1)
        fig.add_trace(go.Bar(
            name=f"{m} mean_L",
            x=xc, y=b2["means"],
            width=BAR_WIDTH,
            marker_color=color,
            opacity=0.75,
            customdata=cd2,
            hovertemplate=(
                f"method={m}<br>"
                "bin=[%{customdata[0]:.1f}, %{customdata[1]:.1f}]<br>"
                "total=%{customdata[2]:.0f}<br>"
                "mean_L=%{customdata[3]:.3f}<extra></extra>"
            ),
        ), row=3, col=1)

    if not any_data:
        return None

    # Layout
    fig.update_layout(
        title=f"All Methods | {sp} | {nt}",
        # 关键：用 stack 让 success/failure 同 method 堆叠，而不同 method 因为 x offset 不同不会堆叠
        barmode="stack",
        legend_title="",
        height=950,
        # 这两个只影响“同一个 x 值的柱子之间”与“不同 x 组之间”的距离
        # 你说 bar 之间太近：可以把 bargap 调大一点（组间距离）
        bargap=0.18,
    )

    # X axis fixed 0~50
    fig.update_xaxes(range=[X_MIN, X_MAX], row=1, col=1)
    fig.update_xaxes(range=[X_MIN, X_MAX], row=2, col=1)
    fig.update_xaxes(range=[X_MIN, X_MAX], row=3, col=1, title_text="episode_length (L*)")

    fig.update_yaxes(title_text="success_rate", range=[0, 1], row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="count", row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text="mean_SPL", row=2, col=1)
    fig.update_yaxes(title_text="mean_L_actual", row=3, col=1)

    return fig

# ----------------------------
# Generate 5 figures total
# ----------------------------
for sp, nt in SITUATIONS:
    fig = make_combined_fig_for_situation(sp, nt)
    if fig is not None:
        fig.show()
    else:
        print(f"[skip] no data for {sp} | {nt}")

Missing: {'no_ep': 0, 'no_d': 0, 'no_Lstar': 0, 'no_Lactual': 0, 'bad_cast': 0, 'x_outside_0_50': 42}


[skip] no data for indoor | place


In [ ]:
#Failure Case
from collections import Counter
import numpy as np

# ----------------------------
# New rule threshold
# ----------------------------
PERCEPTION_ERR_THRESH = 1.5

# ----------------------------
# Reasons: remove stop_called, add perception_error
# ----------------------------
REASON_ORDER = [
    "perception_error",
    "stuck",
    "bad_orientation",
    "time_out",
    "terrain_out_of_bounds",
    "other",
]

# ----------------------------
# Helpers: method / split / nav type
# ----------------------------
def get_method(r):
    return r.get("method") or r.get("method_name") or r.get("model") or "unknown"

def get_split_from_episode_label(ep_label: str):
    if not isinstance(ep_label, str):
        return None
    s = ep_label.strip().lower()
    if s.startswith("innout"):
        return "innout"
    if s.startswith("gr"):
        return "indoor"
    if s.startswith("vc"):
        return "outdoor"
    return "other"

def get_nav_type_from_episode_label(ep_label: str):
    if not isinstance(ep_label, str):
        return None
    s = ep_label.strip().lower()
    return "place" if "_store" in s else "object"

def get_distance_value_for_table(r):
    """
    Try common keys. Change here if your record uses a different key.
    """
    for k in ["distance_to_goal", "final_distance", "dist_to_goal", "d_goal"]:
        if k in r and r[k] is not None:
            try:
                return float(r[k])
            except Exception:
                return None
    return None

def remap_reason_with_perception_rule(r):
    """
    Apply your rule:
    - if stop_called and d < 1.5 => ignore (return None)
    - if stop_called and d >= 1.5 => perception_error
    - else keep original reason (unknown -> other)
    Also: eliminate stop_called entirely from output space.
    """
    raw = r.get("termination_reason", None)
    raw = raw.strip() if isinstance(raw, str) else None

    # normalize unknowns
    known_base = {"stop_called","stuck","bad_orientation","time_out","terrain_out_of_bounds"}
    if raw not in known_base:
        raw = "other"

    if raw == "stop_called":
        d = get_distance_value_for_table(r)
        if d is None:
            # distance missing -> safest: treat as other? or perception_error?
            # Here we map to perception_error to avoid silently dropping data,
            # but you can change to "other" if you prefer.
            return "perception_error"
        if d < PERCEPTION_ERR_THRESH:
            return None  # ignore
        return "perception_error"

    return raw  # stuck/bad_orientation/time_out/terrain_out_of_bounds/other

# ----------------------------
# Build bucket: (method, split, nav_type) -> list[record]
# ----------------------------
methods = sorted(set(get_method(r) for r in records))

bucket = {}
for r in records:
    ep = r.get("episode_label", None)
    if ep is None:
        continue
    m = get_method(r)
    sp = get_split_from_episode_label(ep)
    nt = get_nav_type_from_episode_label(ep)
    bucket.setdefault((m, sp, nt), []).append(r)

# ----------------------------
# Make table for one (split, nav)
# Output: rows = methods, cols = reasons + totals
# Values: percent (and counts)
# ----------------------------
def build_table_for_split_nav(split_name: str, nav_type: str, use_percent=True):
    """
    Returns:
      header: list[str]
      rows: list[list[str]]  (strings ready to print)
    """
    header = ["method"]
    for reason in REASON_ORDER:
        header.append(f"{reason} (%)" if use_percent else f"{reason} (count)")
    header += ["N_used", "N_ignored_stop<1.5"]

    rows = []
    for m in methods:
        sub = bucket.get((m, split_name, nav_type), [])

        counts = Counter()
        ignored = 0
        used = 0

        for rr in sub:
            new_reason = remap_reason_with_perception_rule(rr)
            if new_reason is None:
                ignored += 1
                continue
            counts[new_reason] += 1
            used += 1

        # build row
        row = [m]
        for reason in REASON_ORDER:
            c = counts.get(reason, 0)
            if use_percent:
                val = (c / used * 100.0) if used > 0 else 0.0
                row.append(f"{val:.1f}")
            else:
                row.append(str(c))

        row.append(str(used))
        row.append(str(ignored))
        rows.append(row)

    return header, rows

# ----------------------------
# Pretty print as text table (no pandas)
# ----------------------------
def print_table(header, rows, title=None):
    if title:
        print("\n" + title)
    # column widths
    widths = [len(h) for h in header]
    for r in rows:
        for i, cell in enumerate(r):
            widths[i] = max(widths[i], len(str(cell)))

    def fmt_row(r):
        return " | ".join(str(cell).ljust(widths[i]) for i, cell in enumerate(r))

    print(fmt_row(header))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(fmt_row(r))

# ----------------------------
# Generate 6 tables: 3 splits × 2 nav types
# ----------------------------
SPLITS = ["indoor", "outdoor", "innout"]
NAVS = ["object", "place"]

for sp in SPLITS:
    for nt in NAVS:
        header, rows = build_table_for_split_nav(sp, nt, use_percent=True)
        print_table(header, rows, title=f"[Termination reason table] split={sp} | nav={nt}  (stop_called removed; perception_error added; stop_called&d<1.5 ignored)")


[Termination reason table] split=indoor | nav=object  (stop_called removed; perception_error added; stop_called&d<1.5 ignored)
method        | perception_error (%) | stuck (%) | bad_orientation (%) | time_out (%) | terrain_out_of_bounds (%) | other (%) | N_used | N_ignored_stop<1.5
--------------+----------------------+-----------+---------------------+--------------+---------------------------+-----------+--------+-------------------
modular_agent | 45.7                 | 25.2      | 9.4                 | 19.7         | 0.0                       | 0.0       | 127    | 5                 
poliformer    | 8.2                  | 56.8      | 4.1                 | 30.8         | 0.0                       | 0.0       | 146    | 2                 
uninavid      | 23.4                 | 46.8      | 2.1                 | 27.7         | 0.0                       | 0.0       | 141    | 9                 

[Termination reason table] split=indoor | nav=place  (stop_called removed; perception_error

In [61]:
import json
import numpy as np

THRESH = 1.6  # success = (distance_to_goal < THRESH)

# ----------------------------
# Helpers: split / nav_type (your rule: "_store" => place)
# ----------------------------
def get_split_from_episode_label(ep_label: str):
    """Map episode_label prefix to split name."""
    if not isinstance(ep_label, str):
        return None
    s = ep_label.strip().lower()
    if s.startswith("gr"):
        return "indoor"
    if s.startswith("vc"):
        return "outdoor"
    if s.startswith("innout"):
        return "innout"
    return "other"

def get_nav_type_from_episode_label(ep_label: str):
    """
    Use your latest rule:
    - episode_label contains '_store' -> place navigation
    - otherwise                      -> object navigation
    """
    if not isinstance(ep_label, str):
        return None
    s = ep_label.strip().lower()
    if "_store" in s:
        return "place"
    return "object"

def safe_float(x):
    """Safely cast to float; return None if invalid."""
    try:
        if x is None:
            return None
        if isinstance(x, bool):
            return None
        return float(x)
    except Exception:
        return None

def get_distance_value(r):
    """
    Extract final distance-to-goal from a record.
    Adjust this if your key is different.
    """
    return safe_float(r.get("distance_to_goal"))

def compute_success(r, use_success_field=False):
    """
    Compute success for a record.
    - If use_success_field=True: use r["success"] directly (0/1)
    - Else: success = (distance_to_goal < THRESH)
    """
    if use_success_field:
        s = safe_float(r.get("success"))
        if s is None:
            return None
        return 1.0 if s > 0 else 0.0

    d = get_distance_value(r)
    if d is None:
        return None
    return 1.0 if d < THRESH else 0.0

# ----------------------------
# Load two result.json files
# ----------------------------
def load_records(json_path):
    """Load records from a result.json (list of dicts)."""
    with open(json_path, "r") as f:
        data = json.load(f)
    # Some logs wrap list under a key; handle both
    if isinstance(data, dict):
        # common patterns: {"records": [...]}, {"results": [...]}
        for k in ["records", "results", "episodes"]:
            if k in data and isinstance(data[k], list):
                return data[k]
        raise ValueError(f"Unrecognized dict format in {json_path}: keys={list(data.keys())[:20]}")
    if not isinstance(data, list):
        raise ValueError(f"Expected list in {json_path}, got {type(data)}")
    return data

# ----------------------------
# Index records by episode_label, then intersect
# ----------------------------
def index_by_episode(records):
    """
    Index by episode_label.
    If duplicates exist, keep the last one (can change to assert if needed).
    """
    idx = {}
    for r in records:
        ep = r.get("episode_label")
        if ep is None:
            continue
        idx[ep] = r
    return idx

# ----------------------------
# Aggregate success rate over a subset
# ----------------------------
def success_rate_from_eps(ep_list, idx, use_success_field=False):
    """Compute success rate over given episode labels using idx[ep]."""
    succs = []
    missing = 0
    for ep in ep_list:
        r = idx.get(ep)
        if r is None:
            missing += 1
            continue
        s = compute_success(r, use_success_field=use_success_field)
        if s is None:
            missing += 1
            continue
        succs.append(s)
    if len(succs) == 0:
        return float("nan"), 0, missing
    return float(np.mean(np.array(succs, dtype=float))), len(succs), missing

def filter_eps(ep_list, split=None, nav_type=None):
    """Filter episodes by split and/or nav_type using episode_label string."""
    out = []
    for ep in ep_list:
        sp = get_split_from_episode_label(ep)
        nt = get_nav_type_from_episode_label(ep)
        if split is not None and sp != split:
            continue
        if nav_type is not None and nt != nav_type:
            continue
        out.append(ep)
    return out

# ----------------------------
# Main: compute on common episodes only
# ----------------------------
def compare_two_results_on_common_eps(path_a, path_b, name_a="A", name_b="B", use_success_field=False):
    rec_a = load_records(path_a)
    rec_b = load_records(path_b)

    idx_a = index_by_episode(rec_a)
    idx_b = index_by_episode(rec_b)

    eps_a = set(idx_a.keys())
    eps_b = set(idx_b.keys())
    common = sorted(eps_a.intersection(eps_b))

    print(f"{name_a}: episodes={len(eps_a)}")
    print(f"{name_b}: episodes={len(eps_b)}")
    print(f"COMMON episodes={len(common)}")

    # Define groups you asked for: 4 combos + all merged
    groups = [
        ("indoor",  "object"),
        ("indoor",  "place"),
        ("outdoor", "object"),
        ("outdoor", "place"),
        ("innout",  "object"),
        ("innout",  "place"),
        ("all",     "object"),
        ("all",     "place"),
        ("all",     None),   # truly all merged (place + object)
    ]

    print("\n=== Success Rate on COMMON episodes only ===")
    for sp, nt in groups:
        if sp == "all":
            sub = common
        else:
            sub = filter_eps(common, split=sp, nav_type=None)

        if nt is not None:
            sub = filter_eps(sub, split=None if sp == "all" else sp, nav_type=nt)

        sr_a, n_a, miss_a = success_rate_from_eps(sub, idx_a, use_success_field=use_success_field)
        sr_b, n_b, miss_b = success_rate_from_eps(sub, idx_b, use_success_field=use_success_field)

        label = f"{sp:6s} | {('merged' if nt is None else nt):6s}"
        # n_a and n_b should be identical because sub is common episodes,
        # but we print both in case some records miss distance/success fields.
        print(f"[{label}]  n={len(sub):4d}  {name_a}: SR={sr_a:.3f} (used {n_a}, miss {miss_a})   {name_b}: SR={sr_b:.3f} (used {n_b}, miss {miss_b})")

# ----------------------------
# USAGE
# ----------------------------
path_a = "/home/junzhe_lighthouse/lighthouse/home/junzhe/Projects/SG-VLN/dump2/benchmark_uninavid_jan28/result.json"
path_b = "/home/junzhe_lighthouse/lighthouse/home/junzhe/Projects/SG-VLN/dump2/benchmark_uninavid_vln_jan29/result.json"
compare_two_results_on_common_eps(path_a, path_b, name_a="uninavid", name_b="vln", use_success_field=False)

uninavid: episodes=301
vln: episodes=201
COMMON episodes=200

=== Success Rate on COMMON episodes only ===
[indoor | object]  n= 147  uninavid: SR=0.116 (used 147, miss 0)   vln: SR=0.082 (used 147, miss 0)
[indoor | place ]  n=   0  uninavid: SR=nan (used 0, miss 0)   vln: SR=nan (used 0, miss 0)
[outdoor | object]  n=  23  uninavid: SR=0.000 (used 23, miss 0)   vln: SR=0.000 (used 23, miss 0)
[outdoor | place ]  n=  30  uninavid: SR=0.167 (used 30, miss 0)   vln: SR=0.167 (used 30, miss 0)
[innout | object]  n=   0  uninavid: SR=nan (used 0, miss 0)   vln: SR=nan (used 0, miss 0)
[innout | place ]  n=   0  uninavid: SR=nan (used 0, miss 0)   vln: SR=nan (used 0, miss 0)
[all    | object]  n= 170  uninavid: SR=0.100 (used 170, miss 0)   vln: SR=0.071 (used 170, miss 0)
[all    | place ]  n=  30  uninavid: SR=0.167 (used 30, miss 0)   vln: SR=0.167 (used 30, miss 0)
[all    | merged]  n= 200  uninavid: SR=0.110 (used 200, miss 0)   vln: SR=0.085 (used 200, miss 0)
